In [ ]:
# Root filesystem /tmp is full; keep temporary files on the larger /data volume.
import os
os.makedirs("/data/susanket/tmp", exist_ok=True)
os.environ["TMPDIR"] = "/data/susanket/tmp"

import io
import sys
import tqdm
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision import transforms
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

# Add your project src directory
sys.path.append(r"/home/susanket/esri-vlm/src")
from model import CLIP, build_model
from main import CLIPLightningModule

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
CKPT_PATH = "/home/susanket/esri-vlm/lax2unwh/checkpoints/epoch=98-step=584793.ckpt"  # Update path

ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)

if "hyper_parameters" in ckpt:
    # Clean up non-serializable or unnecessary hyperparameters
    for hp_key in ["eval_downstream", "air_temp_data_path", "election_data_path", "_instantiator"]:
        ckpt["hyper_parameters"].pop(hp_key, None)

    model_hp = dict(ckpt["hyper_parameters"])
    model_hp["load_text_backbone"] = True  # eval encodes raw text queries with Qwen
    lightning_model = CLIPLightningModule(**model_hp).to(device)
    lightning_model.load_state_dict(ckpt["state_dict"])
    lightning_model.eval()
    model = lightning_model.model
    tokenizer = lightning_model.tokenizer
else:
    model = build_model(ckpt["state_dict"]).to(device)
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B", padding_side="left")

model = model.to(device)
model.eval()

# Handle DINOv2 backbone if feature extraction is required
dino_backbone = None
if getattr(model.visual, "precomputed", False):
    from model import DINOV2_REPO
    try:
        dino_backbone = torch.hub.load(DINOV2_REPO, model.visual.model_name, trust_repo=True)
    except TypeError:
        dino_backbone = torch.hub.load(DINOV2_REPO, model.visual.model_name)
    dino_backbone.eval().to(device)

print("Model successfully loaded!")

In [ ]:
PARQUET_PATH = "/data/susanket/vlm/images3/image-00.parquet"  # Update path
IMAGE_COLUMN = "image_bytes"  # Column name containing raw image bytes or dict with 'bytes'

df = pd.read_parquet(PARQUET_PATH).sample(10_000)
print(f"Loaded {len(df)} samples from {PARQUET_PATH}")

image_transform = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def decode_and_transform(raw_image_data):
    """Helper to handle bytes directly or HuggingFace Datasets dict format."""
    if isinstance(raw_image_data, dict) and "bytes" in raw_image_data:
        raw_bytes = raw_image_data["bytes"]
    else:
        raw_bytes = raw_image_data

    try:
        img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
    except Exception:
        img = Image.new("RGB", (224, 224))
    return img

raw_pil_images = [decode_and_transform(row[IMAGE_COLUMN]) for _, row in tqdm.tqdm(df.iterrows())]
tensor_images = torch.stack([image_transform(img) for img in tqdm.tqdm(raw_pil_images)])

In [ ]:
# ==============================================================================
# Cell 3: Image Feature Extraction (Full 256 Dimensions)
# ==============================================================================
BATCH_SIZE = 256
all_img_embeddings = []

with torch.no_grad():
    for i in range(0, len(tensor_images), BATCH_SIZE):
        batch = tensor_images[i : i + BATCH_SIZE].to(device)

        if dino_backbone is not None:
            feats = dino_backbone.forward_features(batch)
            batch = torch.cat([
                feats["x_norm_clstoken"].unsqueeze(1),
                feats["x_norm_patchtokens"]
            ], dim=1)

        embeddings = model.encode_image(batch)
        embeddings = F.normalize(embeddings, dim=-1)
        all_img_embeddings.append(embeddings.cpu().numpy())

raw_image_embeddings = np.concatenate(all_img_embeddings, axis=0)
print(f"Computed raw image embeddings matrix shape: {raw_image_embeddings.shape}")

In [ ]:
# ==============================================================================
# Cell 4: Text Feature Extraction (Full 256 Dimensions)
# ==============================================================================
# QUERIES = [
#     "residential buildings",
#     # "solar panel farm",
#     "Solar Panels",
#     "Red buildings by a river",
#     "Edificios rojos junto a un río",
#     "नदी के किनारे लाल इमारतें",
#     # # "河边的红色建筑",
#     "swimming pool",
#     # "commercial airport runway",
#     # "dense forest and river",
#     # "bseball fields",
#     "baseball fields",
#     # "sports fields",
#     "solar panels",
#     "no solar panels",
#     "red building",
#     "no red building",
#     "red building on the right of a river",
#     "a river throguh the left of a red building"
# ]
QUERIES = [
    "road",
    "river",
    "building",
    "house",
    "forest",
    "Swimming pools",
    'playground'
]

with torch.no_grad():
    enc = tokenizer(
        QUERIES,
        padding=True,
        truncation=True,
        max_length=model.transformer.context_length,
        return_tensors="pt",
    )
    tokens = {k: v.to(device) for k, v in enc.items() if k in ("input_ids", "attention_mask")}

    text_embeddings = model.encode_text(tokens)
    raw_text_embeddings = F.normalize(text_embeddings, dim=-1).cpu().numpy()

print(f"Computed raw text embeddings matrix shape: {raw_text_embeddings.shape}")

In [ ]:
# ==============================================================================
# Cell 5: Truncation, L2 Re-Normalization, & Multi-Dimension Retrieval Plot
# ==============================================================================
import matplotlib.pyplot as plt
import matplotlib as mpl

# Set up global font fallbacks for Chinese, Hindi, and Latin
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = [
    'DejaVu Sans',
    'Noto Sans CJK SC',
    'Noto Sans CJK JP',
    'Noto Sans Devanagari',
    'SimHei',
    'Arial Unicode MS'
]
mpl.rcParams['axes.unicode_minus'] = False

DIMENSIONS = [64, 128, 256]
TOP_K = 5

def get_normalized_slice(embeddings: np.ndarray, dim: int) -> np.ndarray:
    """Slices vector to target dimension and re-normalizes L2 unit length."""
    sub_emb = embeddings[:, :dim]
    norms = np.linalg.norm(sub_emb, axis=1, keepdims=True)
    norms[norms == 0] = 1e-12  # Avoid division by zero
    return sub_emb / norms

# Precompute similarity matrices for 64, 128, and 256 dims
similarity_matrices = {}
for d in DIMENSIONS:
    txt_sliced = get_normalized_slice(raw_text_embeddings, d)
    img_sliced = get_normalized_slice(raw_image_embeddings, d)
    similarity_matrices[d] = txt_sliced @ img_sliced.T

# Plot comparative retrieval results side-by-side
for q_idx, query in enumerate(QUERIES):
    fig, axes = plt.subplots(len(DIMENSIONS), TOP_K, figsize=(18, 3.5 * len(DIMENSIONS)))
    fig.suptitle(f'Query: "{query}"', fontsize=16, fontweight="bold", y=1.02)

    for row_idx, d in enumerate(DIMENSIONS):
        scores = similarity_matrices[d][q_idx]
        top_k_indices = np.argsort(-scores)[:TOP_K]

        for col_idx, img_idx in enumerate(top_k_indices):
            score = scores[img_idx]
            ax = axes[row_idx, col_idx] if len(DIMENSIONS) > 1 else axes[col_idx]

            ax.imshow(raw_pil_images[img_idx])
            ax.set_title(f"Dim {d} | Rank {col_idx+1}\nScore: {score:.4f}", fontsize=10)
            ax.axis("off")

    plt.tight_layout()
    plt.show()

---

In [ ]:
# ==============================================================================
# Cell 5: Create a Collage of N=100 Images with Object Distributions (MRL-style)
# ==============================================================================

import matplotlib.pyplot as plt
import numpy as np

# 1. Compute cosine similarity matrix between image and text embeddings
# Both raw_image_embeddings and raw_text_embeddings are already L2-normalized.
similarity_matrix = np.matmul(raw_image_embeddings, raw_text_embeddings.T) # shape: (10000, len(QUERIES))

# Apply temperature-scaled softmax to get probability distributions over objects
def softmax(x, axis=-1, tau=0.07):
    e_x = np.exp((x - np.max(x, axis=axis, keepdims=True)) / tau)
    return e_x / e_x.sum(axis=axis, keepdims=True)

probs_matrix = softmax(similarity_matrix, axis=-1)

# 2. Select N = 100 random samples from the dataset
N = 100
np.random.seed(42)
selected_indices = np.random.choice(len(raw_pil_images), size=N, replace=False)

# 3. Plotting the grid collage
cols = 10
rows = (N + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(22, 2.2 * rows))
axes = axes.flatten()

for idx, img_idx in enumerate(selected_indices):
    ax = axes[idx]
    img = raw_pil_images[img_idx]
    dist = probs_matrix[img_idx]
    
    # Display the image
    ax.imshow(img)
    ax.axis('off')
    
    # Show top predicted query and probability as the title
    top_query_idx = np.argmax(dist)
    top_prob = dist[top_query_idx]
    query_text = QUERIES[top_query_idx]
    
    ax.set_title(f"{query_text}\n({top_prob:.2f})", fontsize=7, pad=3)

# Hide any unused axes
for idx in range(N, len(axes)):
    axes[idx].axis('off')

plt.suptitle("Collage of N=100 Images with Top Object Predictions and Distribution Scores", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()